In [ ]:
import pandas as pd
churn=pd.read_csv(r"C:\Users\rahma\Downloads\customer_churn_1M.csv")

In [ ]:
churn.info()

In [ ]:
churn.shape

In [ ]:
churn.size

In [ ]:
churn.head()

In [ ]:
churn.sample(10)

In [ ]:
churn.describe()

In [ ]:
churn.isnull().sum()

In [ ]:
churn.isnull().sum().sum()

In [ ]:
churn.duplicated().sum() 

In [ ]:
churn["customer_id"].duplicated().sum()

In [ ]:
churn_rate=round(churn["churn"].mean() * 100,2)  
print(churn_rate)

In [ ]:
for column in churn.columns:
    print(f"\n{column}:")
    print(churn[column].unique())

# 2-Feature selection

In [ ]:
columns_to_drop = ['gender','education','marital_status','senior_citizen','paperless_billing','avg_monthly_gb','credit_score','signup_date',
'has_device_protection' ,'has_streaming_tv','has_streaming_movies','has_phone_service' ,'has_online_backup']

churn= churn.drop(columns=columns_to_drop)

In [ ]:
churn.shape

# 3-Data cleaning

# Handle missing values

In [ ]:
null_percentage= (churn.isnull().mean() * 100).round(1)
print(null_percentage)

In [ ]:
def check_outliers(column):   
    Q1= column.quantile(0.25)
    Q3= column.quantile(0.75)
    
    IQR= Q3 - Q1
    
    lower_bound= Q1 - 1.5 * IQR
    upper_bound= Q3 + 1.5 * IQR
    
    outliers= column[(column < lower_bound) | (column > upper_bound)]
    
    if len(outliers)>0:
        print("There are outliers in ",column.name)
    else:
        print("There are no outliers in ",column.name)

check_outliers(churn["annual_income"])
check_outliers(churn["customer_satisfaction"])
check_outliers(churn["num_complaints"]) 

In [ ]:
churn['annual_income'] = churn['annual_income'].fillna(churn['annual_income'].median())

In [ ]:
churn['customer_satisfaction'] = churn['customer_satisfaction'].fillna(churn['customer_satisfaction'].mean())

In [ ]:
churn['num_complaints'] = churn['num_complaints'].fillna(churn['num_complaints'].median())

In [ ]:
churn.isnull().sum()

In [ ]:
cols = ["has_internet_service","has_online_security","has_tech_support","churn"]
churn[cols] = churn[cols].replace({0: 'No', 1: 'Yes'})

In [ ]:
churn["churn"].unique()

# change dtypes

In [ ]:
churn.info()

In [ ]:
churn['customer_id'] =churn['customer_id'].astype('string')
churn['age'] = churn['age'].astype(int)
churn['contract'] = churn['contract'].astype('string')
churn['payment_method'] = churn['payment_method'].astype('string')
churn['has_internet_service'] = churn['has_internet_service'].astype('string')
churn['has_tech_support'] = churn['has_tech_support'].astype('string')
churn['has_online_security'] = churn['has_online_security'].astype('string')
churn['has_online_security'] = churn['has_online_security'].astype('string')
churn['churn'] = churn['churn'].astype('string')
churn['customer_satisfaction'] = churn['customer_satisfaction'].astype(int)
churn['num_complaints'] = churn['num_complaints'].astype(int)
churn['annual_income'] = churn['annual_income'].astype(float)
churn['monthlycharges'] = churn['monthlycharges'].astype(float)
churn['totalcharges'] = churn['totalcharges'].astype(float)

In [ ]:
churn.info()

# organizing data

In [ ]:
churn=churn.rename(columns={'tenure': 'tenure(months)'})

In [ ]:
churn['has_internet_service'] = churn['has_internet_service'].str.upper()
churn['has_online_security'] = churn['has_online_security'].str.upper()
churn['has_tech_support'] = churn['has_tech_support'].str.upper()
churn['churn'] = churn['churn'].str.upper()
churn['payment_method'] = churn['payment_method'].str.upper()
churn['contract'] = churn['contract'].str.upper()

In [ ]:
churn.head()

In [ ]:
churn['payment_method'] = churn['payment_method'].str.replace('_',' ')
churn['contract'] = churn['contract'].str.replace('_',' ')

In [ ]:
churn.sample(10)

# Handle inconsistent values

In [ ]:
(churn['totalcharges'] < churn['monthlycharges']).sum()

In [ ]:
churn[churn['totalcharges'] < churn['monthlycharges']]['tenure(months)'].value_counts()

In [ ]:
(churn['tenure(months)'] == 1).sum()

In [ ]:
churn.loc[churn['tenure(months)'] == 1, 'totalcharges'] = churn.loc[
    churn['tenure(months)'] == 1, 'monthlycharges'
]

In [ ]:
(churn['totalcharges'] < churn['monthlycharges']).sum()

In [ ]:
churn.loc[churn['totalcharges'] < churn['monthlycharges']]

In [ ]:
churn = churn.drop(churn[churn['totalcharges'] < churn['monthlycharges']].index)

In [ ]:
(churn['totalcharges'] < churn['monthlycharges']).sum()

In [ ]:
churn.to_csv("Final_Projectt.csv") 

# Feature Enginnering

In [ ]:
churn.columns

In [ ]:
churn["Charges_Per_Service"] = churn["monthlycharges"] / churn["num_services"].replace(0, 1)

In [ ]:
churn["Charges_Per_Tenure_Month"] = churn["totalcharges"] / churn["tenure(months)"].replace(0, 1)


In [ ]:
churn["ServiceCalls_Per_Service"] = churn["num_service_calls"] / churn["num_services"].replace(0, 1)

In [ ]:
churn["Age_Group"] = pd.cut(churn["age"],
    bins=[0, 18, 30, 45, 60, 100],
    labels=["Under 18", "Young", "Adult", "Middle Age", "Senior"],)

In [ ]:
churn["Service_Efficiency"] = churn["num_services"] / churn["tenure(months)"].replace(0, 1)

In [ ]:
churn["Satisfaction_Category"] = pd.cut(churn["customer_satisfaction"],
    bins=[-1, 3, 5, 7, 10],
    labels=["Poor", "Average", "Good", "Excellent"],)

In [ ]:
churn["Complaints_Calls_Gap"] = churn["num_complaints"] - churn["num_service_calls"]

In [ ]:
churn["Calls_Per_Tenure_Month"] = churn["num_service_calls"] / churn["tenure(months)"].replace(0, 1)

In [ ]:
churn["Is_Long_Tenure"] = (churn["tenure(months)"] > churn["tenure(months)"].quantile(0.75)).astype(int)

In [ ]:
churn["MonthlyCharges_Category"] = pd.cut(churn["monthlycharges"],
    bins=[0, 35, 70, 120, churn["monthlycharges"].max()],
    labels=["Low", "Medium", "High", "Very High"],)

In [ ]:
churn["Avg_MonthlyCharges_By_Contract"] = churn.groupby("contract")["monthlycharges"].transform("mean")

In [ ]:
churn["Risk_Score"] = (
    (churn["late_payments"] > 0).astype(int) * 30 +
    (churn["customer_satisfaction"] <= 2).astype(int) * 25 +
    (churn["contract"] == 'Month-to-Month').astype(int) * 25 +
    (churn["num_complaints"] > 2).astype(int) * 20
)

In [ ]:
churn["Risk_Level"] = pd.cut(
    churn["Risk_Score"],
    bins=[-1, 25, 55, 100],
    labels=["Low Risk", "Medium Risk", "High Risk"]
)

In [ ]:
churn[["Risk_Score", "Risk_Level"]].head()

In [ ]:
churn["Risk_Level"].value_counts()

In [ ]:
RFM = pd.DataFrame({
    'customer_id': churn['customer_id'],
    'R_Recency': churn['days_since_last_interaction'],
    'F_frequency': churn['num_services'],
    'M_Monetary': churn['totalcharges']
})

RFM

In [ ]:
RFM["R"] = pd.qcut(RFM["R_Recency"], 5, labels=[5, 4, 3, 2, 1])
RFM["F"] = pd.qcut(RFM["F_frequency"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5])
RFM["M"] = pd.qcut(RFM["M_Monetary"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5])
RFM

In [ ]:
RFM["RFM Score"] = (
    RFM["R"].astype(str) + 
    RFM["F"].astype(str) + 
    RFM["M"].astype(str)
).astype(int)

def segmentation(score):
    score = int(score)
    if score <= 155:
        return "Bronze"
    elif score <= 255:
        return "Silver"
    elif score <= 355:
        return "Gold"
    elif score <= 455:
        return "Platinium"
    else:
        return "Taitinium"

RFM["customer segmentation"] = RFM["RFM Score"].apply(segmentation)

RFM

In [ ]:
segmentation_count = RFM["customer segmentation"].value_counts().reset_index()
segmentation_count.columns = ["Segment", "No. customers"]
segmentation_count

In [ ]:
churn

In [ ]:
churn.to_csv("Final_Projecttt.csv") 